# Chapter 06: Capstone Project — Solutions

## End-to-End Data Analysis of the Tips Dataset

This notebook contains the complete solutions for the capstone project.
Each question is followed by the working code and a brief commentary
highlighting key observations from the results.

The analysis integrates techniques from all previous chapters:

- **Pandas** for data loading, exploration, filtering, grouping, and pivot tables
- **Matplotlib** for foundational plotting
- **Seaborn** for statistical visualizations and multi-panel layouts

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the tips dataset
tips = sns.load_dataset('tips')

# Configure plot aesthetics
sns.set_theme(style='whitegrid')
%matplotlib inline

---
## Part 1: Data Exploration

### Q1: Display the first 10 rows and the shape of the dataset

In [ ]:
# Display the first 10 rows
tips.head(10)

In [ ]:
# Shape of the dataset
print(f'Dataset shape: {tips.shape}')
print(f'Number of rows: {tips.shape[0]}')
print(f'Number of columns: {tips.shape[1]}')

The dataset contains 244 records across 7 columns. Each row represents a single
dining transaction with details about the bill, tip, and customer.

### Q2: Get summary statistics and check for missing values

In [ ]:
# Summary statistics for numerical columns
tips.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(tips.isnull().sum())

There are no missing values in the dataset. The average total bill is around
$19.79 with tips averaging about $3.00. Party sizes range from 1 to 6,
with a median of 2 people per table.

### Q3: What are the unique values in each categorical column?

In [ ]:
categorical_cols = ['sex', 'smoker', 'day', 'time']

for col in categorical_cols:
    print(f'{col}: {tips[col].unique().tolist()}')

The dataset covers four days (Thursday through Sunday), two meal times
(Lunch and Dinner), and records whether the customer is a smoker.

### Q4: What is the data type of each column?

In [ ]:
print(tips.dtypes)

The numerical columns (`total_bill`, `tip`) are stored as `float64`,
`size` is `int64`, and the categorical columns use Pandas `category` dtype,
which is memory-efficient for repeated string values.

### Q5: How many records are there for each day of the week?

In [ ]:
print(tips['day'].value_counts())

Saturday has the most records, followed by Sunday. This makes sense as
weekends tend to have more dining activity. Thursday and Friday have
noticeably fewer entries.

---
## Part 2: Data Manipulation

### Q6: Create a new column `tip_percentage`

In [ ]:
tips['tip_percentage'] = tips['tip'] / tips['total_bill'] * 100

tips.head()

The `tip_percentage` column now shows the tip as a proportion of the total bill.
This normalized measure lets us compare tipping generosity across different
bill sizes.

### Q7: What is the average tip percentage by day?

In [ ]:
avg_tip_by_day = tips.groupby('day')['tip_percentage'].mean().sort_values(ascending=False)

print(avg_tip_by_day)

Sunday tends to have the highest average tip percentage, while Saturday
has the lowest. This could reflect different dining occasions: relaxed
Sunday brunches may encourage more generous tipping compared to busy
Saturday evening crowds.

### Q8: Filter for parties of 4 or more and find their average total bill

In [ ]:
large_parties = tips[tips['size'] >= 4]

print(f'Number of large parties (size >= 4): {len(large_parties)}')
print(f'Average total bill for large parties: ${large_parties["total_bill"].mean():.2f}')

Larger parties naturally generate higher total bills. Comparing this
average to the overall dataset mean of approximately $19.79 shows that
parties of four or more spend considerably more per visit.

### Q9: Group by `sex` and `smoker` — calculate mean `total_bill` and `tip`

In [ ]:
grouped = tips.groupby(['sex', 'smoker'])[['total_bill', 'tip']].mean()

print(grouped)

Male smokers tend to have the highest average total bills, which could
indicate larger orders or more expensive menu choices. The tip amounts
follow a similar pattern, though the differences are moderate.

### Q10: Create a pivot table — day vs time, values = tip_percentage, aggfunc = mean

In [ ]:
pivot = pd.pivot_table(
    tips,
    values='tip_percentage',
    index='day',
    columns='time',
    aggfunc='mean'
)

print(pivot)

The pivot table reveals that lunch and dinner tipping rates vary across
days. Some days show no lunch data (Saturday and Sunday in this dataset
are dinner-only), while weekday lunches often have competitive tip
percentages compared to dinners.

---
## Part 3: Visualization

### Q11: Create a histogram of `total_bill` with 20 bins

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(tips['total_bill'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_title('Distribution of Total Bill Amounts', fontsize=14)
ax.set_xlabel('Total Bill ($)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()

The distribution is right-skewed, with most bills falling between $10 and $25.
A few high-value outliers push past $40, representing special occasions or
larger group dinners.

### Q12: Create a scatter plot of `total_bill` vs `tip`, colored by `time`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=tips,
    x='total_bill',
    y='tip',
    hue='time',
    palette='Set1',
    alpha=0.7,
    ax=ax
)

ax.set_title('Total Bill vs Tip Amount by Meal Time', fontsize=14)
ax.set_xlabel('Total Bill ($)', fontsize=12)
ax.set_ylabel('Tip ($)', fontsize=12)

plt.tight_layout()
plt.show()

There is a clear positive relationship between total bill and tip amount.
Dinner transactions dominate the higher end of the bill spectrum, while
lunch transactions cluster in the lower range. The overall trend suggests
customers tip proportionally regardless of meal time.

### Q13: Create a box plot comparing `tip_percentage` across days

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=tips,
    x='day',
    y='tip_percentage',
    palette='pastel',
    order=['Thur', 'Fri', 'Sat', 'Sun'],
    ax=ax
)

ax.set_title('Tip Percentage Distribution by Day', fontsize=14)
ax.set_xlabel('Day of the Week', fontsize=12)
ax.set_ylabel('Tip Percentage (%)', fontsize=12)

plt.tight_layout()
plt.show()

The median tip percentage is fairly consistent across days, hovering around
15-16%. However, the spread differs: Sunday shows more variability with
several high outliers, suggesting that some Sunday diners are notably generous
while others tip closer to the norm.

### Q14: Create a heatmap of the correlation matrix for numerical columns

In [ ]:
# Select numerical columns and compute correlation
numerical_cols = tips.select_dtypes(include='number')
corr_matrix = numerical_cols.corr()

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=1,
    ax=ax
)

ax.set_title('Correlation Matrix of Numerical Variables', fontsize=14)

plt.tight_layout()
plt.show()

The strongest positive correlation is between `total_bill` and `tip` (around 0.68),
confirming that higher bills lead to higher tips. `total_bill` and `size` are also
positively correlated, as larger parties tend to order more. Interestingly,
`tip_percentage` has a weak negative correlation with `total_bill`, meaning
that while absolute tips increase with the bill, the percentage tends to
decrease slightly for very expensive meals.

### Q15: Create a FacetGrid — scatter of `total_bill` vs `tip`, faceted by `day`, colored by `sex`

In [ ]:
g = sns.FacetGrid(
    tips,
    col='day',
    col_order=['Thur', 'Fri', 'Sat', 'Sun'],
    hue='sex',
    palette='Set2',
    col_wrap=2,
    height=4,
    aspect=1.2
)

g.map_dataframe(sns.scatterplot, x='total_bill', y='tip', alpha=0.7)

g.set_axis_labels('Total Bill ($)', 'Tip ($)')
g.set_titles('Day: {col_name}')
g.add_legend(title='Sex')
g.figure.suptitle('Total Bill vs Tip by Day and Sex', fontsize=14, y=1.02)

plt.tight_layout()
plt.show()

The FacetGrid gives us a clearer view of how tipping patterns differ by day
and sex simultaneously. Saturday and Sunday panels contain the most data
points and show the widest range of bills. Male and female customers show
broadly similar tipping behavior within each day, though male customers
appear slightly more often at the higher end of the bill range.

---
## Summary of Key Findings

1. **No missing data** — the Tips dataset is clean and ready for analysis without preprocessing.
2. **Right-skewed bill distribution** — most bills fall between $10 and $25, with a long tail towards higher amounts.
3. **Strong bill-tip correlation** — higher bills produce higher absolute tips (r = 0.68), but the tip percentage slightly decreases for very large bills.
4. **Dinner dominates** — the majority of records are dinner transactions, which tend to have higher bills than lunch.
5. **Weekend patterns** — Saturday has the most records; Sunday shows the most variability in tip percentage.
6. **Smoking and spending** — male smokers tend to have the highest average total bills in the dataset.
7. **Consistent tipping norms** — despite variation, the median tip percentage stays close to 15-16% regardless of day or time.

---

This concludes the capstone project. The analysis demonstrates a complete
data science workflow: loading and exploring data, transforming and aggregating
it for insights, and communicating findings through clear visualizations.